### Proof of concept for complex object definition

Realistic data on relative photometry.

In [1]:
# Get stuff
import matplotlib.pyplot as plt
from math import log10

In [2]:
def myavg(mylist):
    return sum(mylist) / len(mylist)

In [3]:
def mystd(mylist):
    squares = [ x**2 for x in mylist ]
    variance = myavg(squares) - (myavg(mylist))**2
    stdev = variance**0.5
    return stdev

In [4]:
# Define object

class observation:
    def __init__(self):
        self.jd = 0.0        #Julian day
        self.airmass = 0.00  #Airmass (sex zenith angle)
        self.standard = 0.00
        self.comparison = []  # list of comparion star counts
        self.dmag = -99.0
        self.relflux = 0.0

# Test a definition operation
obs = observation()
obs.jd = 2402943.29283
obs.airmass = 1.33
obs.standard = 43455.1
obs.comparison = [ 13254.2, 6228.3, 9051.6, 8535.2, 11248.4, 6393.5 ]

In [5]:
print(obs.airmass, obs.standard, obs.comparison)

1.33 43455.1 [13254.2, 6228.3, 9051.6, 8535.2, 11248.4, 6393.5]


In [6]:
# Show magnitude difference between star and the 
# average of the comparison stars
print(obs.jd, -2.5*log10(obs.standard / myavg(obs.comparison)))

2402943.29283 -1.6952894090957902


In [15]:
def get_data(filename):
    """
    Reads a file and returns a list of observations
    """
    with open(filename, 'r') as f:
        lines = f.readlines()

    datablock = []

    for line in lines:
        if line[0] == '#':
            ;        # Do nothing
        else:
            obs = observation()
            words = line.split()
            obs.jd = float(words[0])
            obs.airmass = float(words[1])
            obs.standard = float(words[2])
            cflux = []                # <- Candidate for refactoring
            for cword in words[3: ]:
                cflux.append(float(cword))      
            obs.comparison = cflux.copy()
            datablock.append(obs)

    return datablock

In [16]:
filename = 'photdata.tsv'
night1 = get_data(filename)

print('There are %d data lines in the file.' % len(night1))
for k, image in enumerate(night1):
    print(' %2d %11.5f %7.1f %7.1f' % 
          (k, image.jd, image.standard, myavg(image.comparison)))

FileNotFoundError: [Errno 2] No such file or directory: 'photdata.tsv'

In [ ]:
A= [50, 100, 150, 200]
for n, value in enumerate(A):
    print(n, value)

In [ ]:
obstimes = [obs.jd for obs in night1]
airmasses = [obs.airmass for obs in night1]
fig, ax = plt.subplots()
ax.plot(obstimes, airmasses, 'ok')
ax.set_ylim(1.4, 0.95)
ax.set_ylabel('Air mass')
ax.set_xlabel('JD')

In [ ]:
#populate the night 1 list with a calculated value
for obs in night1:
    obs.dmag = -2.5*log10(obs.standard / myavg(obs.comparison))

In [ ]:
starphot = [ obs.dmag for obs in night1 ]
fig, ax = plt.subplots()
ax.plot(obstimes, starphot, 'ok')
ax.set_ylabel('Relative magnitude')
ax.set_xlabel('JD')
ax.invert_yaxis()

In [ ]:
# bad points
diffmags = [o.dmag for o in night1]
klist = [k for k, v in enumerate(diffmags) if v > 0]
print(klist)

In [ ]:
# Copy a new list of observations

clean_data = [ o for k,o in enumerate(night1) if k not in klist]
clean_obstimes = [o.jd for o in clean_data]
meancomp = [ obs.dmag for obs in clean_data ]
fig, ax = plt.subplots()
ax.plot(clean_obstimes, meancomp, 'ok')
ax.set_ylabel('Relative magnitude')
ax.set_xlabel('JD')


In [ ]:
# Test what happens with a blank index list.
klist = []
clean_data = [ o for k,o in enumerate(night1) if k not in klist]
clean_obstimes = [o.jd for o in clean_data]
meancomp = [ obs.dmag for obs in clean_data ]
fig, ax = plt.subplots()
ax.plot(clean_obstimes, meancomp, 'ok')
ax.set_ylabel('Relative magnitude')
ax.set_xlabel('JD')


In [ ]:
# Back to cleaned list
diffmags = [o.dmag for o in night1]
klist = [k for k, v in enumerate(diffmags) if v > 0]
print(klist)
clean_data = [ o for k,o in enumerate(night1) if k not in klist]
clean_obstimes = [o.jd for o in clean_data]
meancomp = [ obs.dmag for obs in clean_data ]
fig, ax = plt.subplots()
ax.plot(clean_obstimes, meancomp, 'ok')
ax.set_ylabel('Relative magnitude')
ax.set_xlabel('JD')


In [ ]:
# Rescale differential magnitudes to average
avg_dmag = myavg(meancomp)
for obs in clean_data:
    obs.dmag -= avg_dmag

clean_obstimes = [o.jd for o in clean_data]
meancomp = [ obs.dmag for obs in clean_data ]
fig, ax = plt.subplots()
ax.plot(clean_obstimes, meancomp, 'ok')
ax.set_ylabel('Relative magnitude')
ax.set_xlabel('JD')

In [ ]:
for obs in clean_data:
    obs.relflux = 10.0**(-0.4 * obs.dmag)
    
mean_relflux = [ obs.relflux for obs in clean_data ]

# Standard deviation
sigma = mystd(mean_relflux)
datalabel = r'$\sigma =$ %.3f' % sigma 

fig, ax = plt.subplots()
ax.plot(clean_obstimes, mean_relflux, 'ok', label=datalabel)
ax.set_ylabel('Relative flux')
ax.set_xlabel('JD')
ax.set_ylim(0.9, 1.1)

xline = (min(clean_obstimes), max(clean_obstimes))       # <- Candidate for refactoring
yline = (1.0, 1.0)
ax.plot(xline, yline, 'red')

ax.legend(loc="upper right")